# YOLO (Ultralytics) training — **no S3 required**

Data is expected **on local disk** next to the repo (EBS on **EC2 g6.xlarge**, your PC, or a Docker volume). **Object storage is not required.**

**Default workflow**
1. On the GPU host: `git clone` → `cd PROJECTS/CrowdNav` → `pip install -r requirements.txt`.
2. Place or generate `data/processed/splits/` with `data.yaml` and `path: .`.
3. Run: `python scripts/train_yolo.py --data-yaml data/processed/splits/data.yaml --epochs 100 --batch 16 --workers 4 --device 0 --model-cfg yolov8m.pt`

**Docker:** build from `deploy/Dockerfile` (see `deploy/docker-compose.yml`).

**Optional:** ClearML via `../.env`. Optional cells below use **SageMaker only if** your org provides managed training and storage; skip them when training on EC2 with disk only.

In [ ]:
# Core: local training + optional ClearML. SageMaker is optional (install only if needed).
%pip install python-dotenv clearml -q

import os
from dotenv import load_dotenv

load_dotenv('../.env')
print("CLEARML keys set:", bool(os.environ.get("CLEARML_API_ACCESS_KEY")))

### 1. Data on disk (no bucket)

- Put `data/processed/splits/` on the training machine (clone + DVC pull, `scp`, rsync, or copy from shared NFS). `data.yaml` uses `path: .` so it works on any path.
- **g6.xlarge / L4**: default hyperparameters match `batch=16`, `workers=4` (16 GB system RAM).
- **ClearML**: keys from `../.env` (optional).

In [ ]:
### 2. Run training (shell on the GPU host)

From `PROJECTS/CrowdNav` on the instance (data already on disk, **no S3**):

```bash
python scripts/train_yolo.py \
  --data-yaml data/processed/splits/data.yaml \
  --epochs 100 --batch 16 --workers 4 --device 0 --model-cfg yolov8m.pt \
  --name crowdnav_g6_v1
```

Jupyter: prefix lines with `!` or use `%%bash` in a code cell.

### 3. (Optional) SageMaker or other managed services

This repo is **not** built around a specific cloud bucket. If you use SageMaker in an environment that **does** supply a data channel, point `--data-yaml` at the mounted path (e.g. under `/opt/ml/input/...`). Otherwise, prefer **EC2 g6 + EBS** and the shell command in the previous cell.

In [ ]:
**Without S3:** training uses only **local or EBS** paths. If you later use a platform that mounts data under a fixed path, set `--data-yaml` to that `data.yaml` (same layout as `data/processed/splits/`).